# Chapter 2

In this chapter we will be working with [**Brain Imaging Datastructure (BIDS)**](https://bids-specification.readthedocs.io/en/stable/), it was built to organise *MRI* datasets and make them manageable and recenlty it has been amended for *EEG/MEG* datastorage as well.
Luckily, *MNE* has [**MNE-BIDS**](https://mne.tools/mne-bids/stable/index.html), which enables *MNE* to interact (read/write) with *BIDS* data or convert to *BIDS* compatible data.


## Libraries & Config

In [ ]:
import matplotlib
import pathlib

import mne

matplotlib.use('QtAgg')

In [ ]:
import mne_bids

# Read Raw Data

In [ ]:
sample_data_dir = mne.datasets.sample.data_path("./")
sample_data_dir = pathlib.Path(sample_data_dir)

raw_path = sample_data_dir / "MEG"  / "sample" / "sample_audvis_raw.fif"
raw_data = mne.io.read_raw(raw_path)

events = mne.find_events(raw_data)
event_id = {
    'Auditory/Left': 1,
    'Auditory/Right': 2,
    'Visual/Left': 3,
    'Visual/Right': 4,
    'Smily': 5,
    'Button': 32
}

# Write the raw data to BIDS

Now BIDS requires the data to also specify at what *power line frequency* in hertz the data was recorded. 

In our case, it is not already stated in the raw data, but, since we know the data was recorded in America and the *power line frequency* is *60 hertz*, therefore, we can add this info at `.info['line_freq']` attribute of our mne data object.

In [ ]:
raw_data.info['line_freq'] = 60
raw_data.info

Now, before writing the raw data to *BIDS* we also have to make sure to add any other necessary info regarding this data that later after *BIDS* conversion could be useful for analysis. For example, our raw data does not have any info regarding the subject and therefore we can look up [`mne.info` guide](https://mne.tools/stable/generated/mne.Info.html) to find which attribute to add such info, which is `subject_info`:
> `subject_info` dict:
>
>   id : int
>       Integer subject identifier.
>
>   his_id : str
>       String subject identifier.
>
>   last_name : str
>       Last name.
>
>   first_name : str
>       First name.
>
>   middle_name : str
>       Middle name.
>
>   birthday : datetime.date
>       The subject birthday.
>>  Changed in version 1.8: This is stored as a date object instead of a tuple of seconds/  microseconds.
>
>   sex : int
>       Subject sex (0=unknown, 1=male, 2=female).
>
>   hand : int
>       Handedness (1=right, 2=left, 3=ambidextrous).
>
>   weight : float
>       Weight in kilograms.
>
>   height : float
>       Height in meters.

In [ ]:
from datetime import date

# create subject info dir
subject_info = {
    'birthday' : date(1999, 1, 1),
    'sex' : 1,
    'hand' : 3
}

# add subject info to raw data info
raw_data.info['subject_info'] = subject_info
raw_data.info

Now to write our *raw data* and also its accompanying *events* to *BIDS*:

1. We have to first create a *BIDS_path* object using the `mne_bids.BIDSPATH()` class (it basically specifies the locations of *BIDS* file) and pass the data entities (*BIDS* folder and file naming scheme) as parameters.

In [ ]:
# let's first create a output dir for the bids data
out_dir = pathlib.Path("./out_data/sample_BIDS")

# BIDSPATH is the main BIDS object
bids_path = mne_bids.BIDSPath(
    # entity values
    subject="01", # subject/participant ID
    session="01", # session ID (optional when there is only one session)
    task="audiovisual", # task name
    run="01", # acquisition run number (optional when there is only one run)
    root=out_dir # directory to store the BIDS data
)


2. After creating the *BIDSPath*, we can call the `mne_bids.write_raw_bids()` method and pass the `raw`, `bids_path`, `events` and `event_id` (if the don't pass the latter two params no events/triggers will be written along the data).
> Note: Often time when storing such data we want to remove key info about the subjects which can link back to them and to do so we can pass the `anonymize` argument. It simply allows you to shift the recording date and remove info such as birthdate and others.

In [ ]:
mne_bids.write_raw_bids(
    raw=raw_data,
    bids_path=bids_path,
    events=events,
    event_id=event_id,
    overwrite=True
)

Now we have our first *BIDS* dataset, but we are still missing some more info because if our data was only *EEG* data then we would have been all fine here, but since, our data also consist of *MEG* that was acquired with *MEGIN / Elekta / NeuroMag* therefore we have to add thier fine-calibration and crosstalk files as well.

## Write MEGIN / Elekta / NeuroMag fine-calibration and crosstalk files
To add these file we simplt have to call the `mne_bids.write_meg_calibration()` and `mne_bids.write_meg_crosstalk()` methods and pass the *BIDSPath* object and the respective file path:

In [ ]:
# path to fine-calibration and crosstalk files
f_calib_path = sample_data_dir / 'SSS' / 'sss_cal_mgh.dat'
c_talk_path = sample_data_dir / 'SSS' / 'ct_sparse_mgh.fif'

mne_bids.write_meg_calibration(
    calibration=f_calib_path,
    bids_path=bids_path
)

mne_bids.write_meg_crosstalk(
    fname=c_talk_path,
    bids_path=bids_path
)

Let's write the same raw data again, but this time act as if it were from a different subject i.e., subject="02"

In [ ]:
# create a BIDSPath object
bids_path_s2 = mne_bids.BIDSPath(
    subject="02",
    session="01",
    task="audiovisual",
    run="01",
    root=out_dir
)

# write bids data for subject 2
mne_bids.write_raw_bids(
    raw=raw_data,
    bids_path=bids_path_s2,
    events=events,
    event_id=event_id,
    overwrite=True
)

# write meg calibration and crosstalk for subject 2
mne_bids.write_meg_calibration(
    calibration=f_calib_path,
    bids_path=bids_path_s2
)
mne_bids.write_meg_crosstalk(
    fname=c_talk_path,
    bids_path=bids_path_s2
)

## Print dir tree and data summary
`mne-bids` come with `.print_dir_tree()` function that print the created file & directory structure for us, we just need to pass the dir path to our *BIDS* data:  

In [ ]:
mne_bids.print_dir_tree(out_dir)

`mne_bids` also comes with a in-built `.make_report()` function which automatically generates a data summary:

In [ ]:
print(mne_bids.make_report(out_dir))

## Reading BIDS data
Let's try to read back the data we just stored according to the BIDS specification. To do that, we need to create a `BIDSPath` object and pass it to `mne_bids.read_raw_bids()`:

In [ ]:
bids_root = pathlib.Path("./out_data/sample_BIDS")

bids_path_read = mne_bids.BIDSPath(
    subject="01",
    session="01",
    task="audiovisual",
    run="01",
    datatype="meg", # meg because when thier are multiple modalities (like meg, eeg) we also specify meg there
    root=bids_root
) 

raw_bids_data = mne_bids.read_raw_bids(bids_path_read)

In [ ]:
raw_bids_data.plot()

## Events are stored as annotations
When we passed the events and event ids when created BIDS data, it automatically store/mark all the events as annotations

In [ ]:
raw_bids_data.annotations[0]

Now to extract the events for later use, such as, when cutting the data into epochs, *mne* would require the events argument and we can extract them by calling `mne` function `.events_from_annotations()` and pass the *BIDS* data to it:

In [ ]:
mne.events_from_annotations?

In [ ]:
events, event_id = mne.events_from_annotations(raw_bids_data)

To visualise how many and in what order the events occured through out the experiment, we can use the *mne* provided function `mne.viz.plot_events()` and pass `events`, `events_id`, and also the `sfreq` which specifies the sampling rate to have the time-axis in seconds and it is stored at `.info['sfreq']` attribute of the data object.

> Sampling is the process to change the Continuous time signal to discrete time signal. Sampling frequency is number of samples taken from the input signal in 1 second. It is also reciprocal of time difference between one sample to the next sample. <br />
So, sampling frequency (or rate) is how often a continuous signal (like sound or light) is measured or "sampled" to convert it into digital data, measured in samples per second (Hertz, Hz). A higher frequency means more snapshots are taken, leading to a more accurate digital representation of the original signal, crucial for capturing detail, like in audio (44.1kHz for CDs). It must be at least twice the highest frequency in the signal (Nyquist Theorem) to prevent information loss, a problem called aliasing. 

In [ ]:
mne.viz.plot_events(
    events=events,
    event_id=event_id,
    sfreq=raw_bids_data.info['sfreq']
)

## Finding the MEG fine-calibration and crosstalk files
To get the path to the fine-calibration and crosstalk files (you might need them to perform maxwell filtering), we just need to call the `.meg_calibration_fpath` & `.meg_crosstalk_fpath` attributes of the `BIDSPath` instance we created when reading the *BIDS* object:

> Maxwell filtering is a powerful technique, primarily for MEG (magnetoencephalography), that uses Maxwell's equations to separate neural signals from magnetic interference, effectively acting like a virtual shielded room to clean up data, correct for head movement, and reconstruct bad channels by separating internal (brain) signals from external (noise) fields

In [ ]:
bids_path.meg_calibration_fpath

In [ ]:
bids_path.meg_crosstalk_fpath

# The End & Have a great day!